# 03-1: LightGBM モデル

M5 Forecasting - Accuracy コンペティション

- **モデル**: LightGBM (Gradient Boosting)
- **評価指標**: WRMSSE → 簡易代替としてRMSLE/RMSEを使用
- **NaN戦略**: NaN(デフォルト) — LightGBMはNaN対応
- **CV戦略**: 時系列5-fold CV（28日window、全モデル統一）

In [ ]:
import sys
import warnings
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import lightgbm as lgb
from sklearn.metrics import mean_squared_error

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'MS Gothic'
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['axes.unicode_minus'] = False

# プロジェクトパス
sys.path.append('../../..')
sys.path.append('../..')
from src.features import build_features, prepare_train_data, get_val_folds, get_feature_cols

INTERMEDIATE_DIR = Path('./intermediate')

def rmsle(y_true, y_pred):
    """Root Mean Squared Logarithmic Error"""
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred)))

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

print('セットアップ完了')

## 1. データ読み込みと特徴量生成

In [ ]:
# 特徴量生成（初回は数分かかる、2回目以降はキャッシュ）
df, feature_cols, val_folds = build_features(use_cache=True)

# 学習データの準備（ラグNaN行を除外）
df_train = prepare_train_data(df, feature_cols)

print(f'\n特徴量数: {len(feature_cols)}')
print(f'特徴量一覧: {feature_cols}')

## 2. LightGBM パラメータ設定

In [ ]:
lgb_params = {
    'objective': 'tweedie',         # ゼロが多い販売データに適合
    'tweedie_variance_power': 1.1,
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 127,
    'min_child_samples': 50,
    'max_depth': -1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'n_estimators': 2000,
    'random_state': 42,
    'verbose': -1,
    'n_jobs': -1,
}

print('LightGBM パラメータ:')
for k, v in lgb_params.items():
    print(f'  {k}: {v}')

## 3. 時系列CV評価

In [ ]:
cv_scores = []
best_iterations = []
importances = []

for fold_info in val_folds:
    fold = fold_info['fold']
    val_start = pd.Timestamp(fold_info['val_start'])
    val_end = pd.Timestamp(fold_info['val_end'])
    
    # 時系列分割
    train_mask = df_train['date'] < val_start
    valid_mask = (df_train['date'] >= val_start) & (df_train['date'] <= val_end)
    
    X_train = df_train.loc[train_mask, feature_cols]
    y_train = df_train.loc[train_mask, 'sales']
    X_valid = df_train.loc[valid_mask, feature_cols]
    y_valid = df_train.loc[valid_mask, 'sales']
    
    print(f'\n--- Fold {fold} ---')
    print(f'  Train: {X_train.shape[0]:,} rows, Valid: {X_valid.shape[0]:,} rows')
    print(f'  期間: ~ {val_start.date()} | {val_start.date()} ~ {val_end.date()}')
    
    # LightGBM Dataset
    dtrain = lgb.Dataset(X_train, label=y_train)
    dvalid = lgb.Dataset(X_valid, label=y_valid, reference=dtrain)
    
    # 学習
    model = lgb.train(
        {k: v for k, v in lgb_params.items() if k != 'n_estimators'},
        dtrain,
        num_boost_round=lgb_params['n_estimators'],
        valid_sets=[dvalid],
        callbacks=[
            lgb.early_stopping(100, verbose=False),
            lgb.log_evaluation(0),
        ],
    )
    
    # 予測 & 評価
    pred = model.predict(X_valid)
    pred = np.clip(pred, 0, None)
    score = rmse(y_valid, pred)
    score_rmsle = rmsle(y_valid, pred)
    
    cv_scores.append({
        'fold': fold, 'rmse': score, 'rmsle': score_rmsle,
        'best_iter': model.best_iteration,
        'val_start': str(val_start.date()), 'val_end': str(val_end.date()),
    })
    best_iterations.append(model.best_iteration)
    
    # 特徴量重要度
    imp = pd.DataFrame({
        'feature': feature_cols,
        'importance': model.feature_importance(importance_type='gain'),
    })
    imp['fold'] = fold
    importances.append(imp)
    
    print(f'  RMSE: {score:.4f}, RMSLE: {score_rmsle:.5f}, best_iter: {model.best_iteration}')

# CV結果サマリー
cv_df = pd.DataFrame(cv_scores)
print(f'\n=== CV結果サマリー ===')
print(f'RMSE  平均: {cv_df["rmse"].mean():.4f} ± {cv_df["rmse"].std():.4f}')
print(f'RMSLE 平均: {cv_df["rmsle"].mean():.5f} ± {cv_df["rmsle"].std():.5f}')
print(f'Best iteration 平均: {np.mean(best_iterations):.0f}')
print(cv_df.to_string(index=False))

## 4. 特徴量重要度

In [ ]:
# Fold平均の特徴量重要度
imp_all = pd.concat(importances)
imp_mean = imp_all.groupby('feature')['importance'].mean().sort_values(ascending=False).reset_index()

# V10: 特徴量重要度の偏り検出
cumsum = imp_mean['importance'].cumsum() / imp_mean['importance'].sum()
top2_share = cumsum.iloc[1]
top5_share = cumsum.iloc[4]
print(f'Top 2 特徴量のシェア: {top2_share*100:.1f}%')
print(f'Top 5 特徴量のシェア: {top5_share*100:.1f}%')
if top2_share > 0.7:
    print('⚠ 重要度が極端に偏っている → チューニングより特徴量追加が有効')

# 可視化
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(imp_mean['feature'][::-1], imp_mean['importance'][::-1], color='steelblue')
ax.set_title('LightGBM 特徴量重要度（Gain平均）')
ax.set_xlabel('平均Gain')
plt.tight_layout()
plt.show()

## 5. 全データ再学習 & 予測サニティチェック

In [ ]:
# Lesson 6: 全データ（train+valid）で再学習、iterations数は固定
best_iter = int(np.mean(best_iterations))
print(f'全データ再学習: iterations = {best_iter}')

X_all = df_train[feature_cols]
y_all = df_train['sales']

dtrain_all = lgb.Dataset(X_all, label=y_all)
final_model = lgb.train(
    {k: v for k, v in lgb_params.items() if k != 'n_estimators'},
    dtrain_all,
    num_boost_round=best_iter,
)

# V4: 予測サニティチェック（学習データに対する予測）
train_pred = final_model.predict(X_all)
train_pred = np.clip(train_pred, 0, None)

checks = {
    'pred_min >= 0': train_pred.min() >= 0,
    'pred_max < 10x train_max': train_pred.max() < 10 * y_all.max(),
    'pred_mean within 2x train_mean': abs(train_pred.mean() - y_all.mean()) < 2 * y_all.mean(),
    'no NaN in predictions': np.isnan(train_pred).sum() == 0,
}
print('\n=== V4: 予測サニティチェック ===')
for check, passed in checks.items():
    status = 'PASS' if passed else 'FAIL'
    print(f'  [{status}] {check}')

# V11: 過学習診断
train_rmse = rmse(y_all, train_pred)
cv_rmse = cv_df['rmse'].mean()
overfit_ratio = train_rmse / cv_rmse
print(f'\n=== V11: 過学習診断 ===')
print(f'  Train RMSE: {train_rmse:.4f}')
print(f'  CV RMSE:    {cv_rmse:.4f}')
print(f'  比率: {overfit_ratio:.3f} (1.0に近いほど良い)')

## 6. 中間データ保存

In [ ]:
results = {
    'model_name': 'LightGBM',
    'params': lgb_params,
    'cv_scores': cv_df.to_dict('records'),
    'cv_mean_rmse': cv_df['rmse'].mean(),
    'cv_std_rmse': cv_df['rmse'].std(),
    'cv_mean_rmsle': cv_df['rmsle'].mean(),
    'cv_std_rmsle': cv_df['rmsle'].std(),
    'best_iteration': best_iter,
    'feature_importance': imp_mean.to_dict('records'),
    'overfit_ratio': overfit_ratio,
}

with open(INTERMEDIATE_DIR / '03-1_lgbm_results.pkl', 'wb') as f:
    pickle.dump(results, f)

print('03-1_lgbm_results.pkl を保存しました')
print(f'\n最終結果: CV RMSE = {cv_df["rmse"].mean():.4f} ± {cv_df["rmse"].std():.4f}')